# GĐ2 v4 — FedAvg là phương pháp chính
FedAvg/MobileNetV3 trên 9 điều kiện × 3 seed. Centralized và Local-only là đối chứng của cùng protocol.
Không dùng notebook v3 hoặc stage1_compat để tạo kết quả GĐ2 mới. Điểm GĐ1 cũ chỉ là tham chiếu lịch sử: split, K và optimizer khác, có rò rỉ nhóm lá.
Notebook mặc định audit và lập kế hoạch; chọn ACTION sau khi kiểm tra plan và chốt ngân sách bằng calibration trên GPU.
Không có trần accuracy. Feature skew hiện là profile ảnh tổng hợp theo client, chưa phải phân chia miền đặc trưng theo Dirichlet.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess

# Điền đúng hai đường dẫn dataset đã gắn vào notebook.
PACKAGE_SOURCE = Path('/kaggle/input/REPLACE_PACKAGE_DATASET/gd2_federated_learning')
DATASET_ROOT = Path('/kaggle/input/REPLACE_PLANTVILLAGE_DATASET/color')
PACKAGE_ROOT = Path('/kaggle/working/gd2_federated_learning')
assert (PACKAGE_SOURCE / 'fl_training/stage2_protocol.py').is_file(), 'Upload package v4 mới nhất'
assert DATASET_ROOT.is_dir(), 'Điền thư mục chứa trực tiếp 38 lớp ảnh'
assert len([p for p in DATASET_ROOT.iterdir() if p.is_dir()]) == 38

if not PACKAGE_ROOT.exists():
    for folder in ('src', 'fl_training', 'configs', 'scripts'):
        shutil.copytree(PACKAGE_SOURCE / folder, PACKAGE_ROOT / folder)
    shutil.copytree(PACKAGE_SOURCE / 'data/partitions_train_v4_content_aware', PACKAGE_ROOT / 'data/partitions_train_v4_content_aware')
    shutil.copy2(PACKAGE_SOURCE / 'data/source_content_cache.json', PACKAGE_ROOT / 'data/source_content_cache.json')
    for name in ('pyproject.toml', 'requirements-kaggle.txt'):
        shutil.copy2(PACKAGE_SOURCE / name, PACKAGE_ROOT / name)
else:
    # Refuse to silently use an older working copy.
    for folder in ('src', 'fl_training'):
        for source in (PACKAGE_SOURCE / folder).rglob('*.py'):
            target = PACKAGE_ROOT / source.relative_to(PACKAGE_SOURCE)
            assert target.is_file() and target.read_bytes() == source.read_bytes(), f'Stale package: {target}'
os.chdir(PACKAGE_ROOT)
sys.path.insert(0, str(PACKAGE_ROOT))
for key, value in {'TORCH_HOME':'/kaggle/working/torch-cache', 'MPLCONFIGDIR':'/kaggle/working/mpl-cache', 'RAY_TMPDIR':'/kaggle/working/ray-temp'}.items():
    Path(value).mkdir(parents=True, exist_ok=True)
    os.environ[key] = value
def cli(*args):
    subprocess.run([sys.executable, '-m', 'fl_training.cli', *args], check=True, cwd=PACKAGE_ROOT)


## Môi trường
Cài dependency từ requirements-kaggle.txt của chính package nếu session chưa có. Giữ nguyên PyTorch CUDA của môi trường khi phù hợp; không dùng venv CPU local để kết luận GPU/AMP đã được kiểm chứng.


In [ ]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-kaggle.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], check=True)
import yaml, torch, torchvision
assert torch.__version__.split('+')[0].startswith('2.6.'), 'Protocol đã kiểm thử với torch 2.6.x'
assert torchvision.__version__.split('+')[0].startswith('0.21.'), 'Protocol đã kiểm thử với torchvision 0.21.x'
assert torch.cuda.is_available(), 'Bật GPU Kaggle cho lượt nghiên cứu chính'


In [ ]:
# Một ngân sách chung cho FedAvg và đối chứng, chốt trước khi xem test.
ROUNDS = 20  # Giá trị dự kiến; cập nhật sau calibration, trước khi chạy main.
runtime = PACKAGE_ROOT / 'configs/runtime_v4'
runtime.mkdir(exist_ok=True)
base = yaml.safe_load((PACKAGE_ROOT / 'configs/train_fedavg_v4.yaml').read_text())
base['data']['dataset_root'] = str(DATASET_ROOT)
base['runtime']['client_device'] = 'cuda'
base['federation']['max_rounds'] = ROUNDS
base_path = runtime / 'base.yaml'
base_path.write_text(yaml.safe_dump(base))
runtime_specs = {}
for name in ('stage2_fedavg_main_v4', 'stage2_controls_v4', 'sweep_content_aware_v4'):
    spec = yaml.safe_load((PACKAGE_ROOT / f'configs/{name}.yaml').read_text())
    spec['base_config'] = str(base_path)
    # New round budget uses a separate namespace, never mixed with prior results.
    spec['sweep_id'] += f'_rounds{ROUNDS}'
    path = runtime / f'{name}.yaml'
    path.write_text(yaml.safe_dump(spec))
    runtime_specs[name] = path


In [ ]:
# Rehash real images, reject changed bytes, then validate configuration.
subprocess.run([sys.executable, 'scripts/audit_content_aware.py',
    '--index', 'data/partitions_train_v4_content_aware/index.json',
    '--cache', 'data/source_content_cache.json', '--dataset-root', str(DATASET_ROOT)], check=True)
cli('preflight', '--config', str(base_path))
cli('sweep', '--config', str(runtime_specs['stage2_fedavg_main_v4']))


In [ ]:
# plan: no training; fedavg: primary experiment; controls: baselines only;
# collect: combine completed results without retraining any method.
ACTION = 'plan'
routes = {'fedavg': ('stage2_fedavg_main_v4', '--execute'),
          'controls': ('stage2_controls_v4', '--execute'),
          'collect': ('sweep_content_aware_v4', '--collect')}
assert ACTION in {'plan', *routes}
if ACTION != 'plan':
    spec_name, flag = routes[ACTION]
    cli('sweep', '--config', str(runtime_specs[spec_name]), flag)


## Diễn giải kết quả
- So sánh FedAvg IID/α=100 với α=1 và α=0,1; các trục quantity/feature giữ cùng trainer và tập đánh giá.
- Sau controls và collect, báo accuracy, macro-F1, trung bình/độ lệch qua seed và chênh lệch Centralized−FedAvg. Giữ cả trường hợp FedAvg không kém hơn, không chọn seed theo test.
- Không lấy điểm v4 trừ trực tiếp số ~99% của GĐ1 cũ. Cần rerun implementation GĐ1 theo cùng protocol để có phép so sánh trực tiếp.
- Chống trùng byte và nhóm lá đã biết không chứng minh loại hết ảnh gần trùng/nhóm lá chưa có metadata. Điểm cao vẫn có thể hợp lệ; kiểm tra holdout thực địa nếu cần đánh giá khả năng tổng quát hóa.
- Sao lưu toàn bộ runs/stage2_fedavg và configs/runtime_v4 trước khi kết thúc session. Sweep này không tự điều phối quota hoặc resume nhiều session; resume từng checkpoint giữ nguyên ngân sách/source/protocol.
